# Day 3 — real v1 dataset → first QLoRA run → first base-vs-tuned numbers

The midweek gate. This notebook (GPU runtime):

1. Loads **CRAPII** (real student essays, `NAME` tagged, emails/phones as real negatives).
2. Builds a **v1 training set** = CRAPII real slice + Faker pattern-type negatives (+ optional
   teacher-distilled synthetic if you set an API key), de-leaked against the quarantined eval.
3. Runs the **real QLoRA** training (r=32/α=32, lr 2e-4, seq 2048, 3 epochs, completion-only).
4. Scores **base vs tuned** on the 51-item quarantined hard cases and prints the delta.

No teacher API key required for the core run (CRAPII + Faker). The synthetic-bulk cell is optional.

> Hard rule (AGENTS.md): the eval set is quarantined — `deleak_and_split` drops any training row
> whose text matches an eval input, so nothing leaks."

## 0. Setup (clone repo + install)

In [ ]:
import os, sys

if not os.path.isdir("/content/slm-deid"):
    !git clone https://github.com/f15cubing/slm-deid.git /content/slm-deid
os.chdir("/content/slm-deid")
!git pull
sys.path.insert(0, os.getcwd())
print("cwd:", os.getcwd())

!pip install -q "unsloth[colab-new] @ git+https://github.com/unslothai/unsloth.git" 2>/dev/null || pip install -q unsloth
!pip install -q trl datasets faker pyyaml kaggle

## 1. Download CRAPII from Kaggle

Upload your `kaggle.json` (Kaggle → Settings → API → Create New Token). Skip if `data/raw` already
has the file.

In [ ]:
import glob, os

if not glob.glob("data/raw/*.json*"):
    from google.colab import files
    files.upload()  # pick kaggle.json
    !mkdir -p ~/.kaggle && cp kaggle.json ~/.kaggle/ && chmod 600 ~/.kaggle/kaggle.json
    !mkdir -p data/raw
    !kaggle datasets download -d langdonholmes/cleaned-repository-of-annotated-pii -p data/raw --unzip

crapii_path = glob.glob("data/raw/*.json*")[0]
print("CRAPII file:", crapii_path)
!ls -la data/raw

## 2. (Optional) teacher-distilled synthetic bulk

Only runs if you set an API key. Leave the key unset to skip and build the dataset from CRAPII +
Faker alone. Weighted toward the hard ambiguous categories.

In [ ]:
import os

# To enable synthetic bulk, set ONE of these, then re-run:
# os.environ["ANTHROPIC_API_KEY"] = "sk-ant-..."
# os.environ["OPENAI_API_KEY"] = "sk-..."

synthetic = []
if os.environ.get("ANTHROPIC_API_KEY") or os.environ.get("OPENAI_API_KEY"):
    from src.datagen.teacher import TeacherGenerator
    from src.datagen.quality_gate import filter_examples
    if os.environ.get("ANTHROPIC_API_KEY"):
        from src.eval.judge import build_anthropic_complete as build_complete
    else:
        from src.eval.judge import build_openai_complete as build_complete
    complete = build_complete()
    teacher = TeacherGenerator(gen=complete, verify=complete)

    # Start modest — ~104 items (~208 sequential API calls, a few minutes). Scale up later.
    counts = {"person_vs_eponym": 20, "person_vs_place": 20, "person_vs_common": 20,
              "first_name_only": 12, "possessive": 12, "third_party": 12, "negative_trap": 8}
    total = sum(counts.values())
    raw, vts, done = [], [], 0
    for cat, n in counts.items():
        for i in range(n):
            ex = teacher.generate(cat, id_=f"gen-{cat}-{i:04d}")
            raw.append(ex); vts.append(teacher.verify_tagging(ex.input))
            done += 1
            if done % 5 == 0:
                print(f"{done}/{total} generated...")  # heartbeat so it doesn't look hung
    synthetic, drops = filter_examples(raw, vts)
    print("synthetic kept:", len(synthetic), "drops:", drops)
else:
    print("No teacher key set — skipping synthetic bulk (CRAPII + Faker only).")

## 3. Assemble v1 dataset (CRAPII slice + Faker negatives + synthetic) → de-leak → split

In [ ]:
from src.datagen.real_data import load_crapii
from src.datagen.negatives import generate_negatives
from src.datagen.generate import deleak_and_split
from src.common.schema import write_jsonl

synthetic = globals().get("synthetic", [])  # empty if the optional teacher cell (§2) was skipped

# Small real slice (keep modest per the NAME_STUDENT/NAME labeling caveat) + Faker negatives.
crapii = load_crapii(crapii_path, names_only=True, max_chars=2000, limit=400)
negatives = generate_negatives(n=200, seed=0)
pool = crapii + negatives + synthetic
print(f"pool: crapii={len(crapii)} negatives={len(negatives)} synthetic={len(synthetic)} total={len(pool)}")

train, val, n_leak = deleak_and_split(pool, eval_dir="eval", val_frac=0.1, seed=0)
write_jsonl("data/splits/train.jsonl", train)
write_jsonl("data/splits/val.jsonl", val)
print(f"train={len(train)} val={len(val)} eval_leak_dropped={n_leak}")

## 4. QLoRA training (real: 3 epochs, r=32/α=32, completion-only)

In [ ]:
from src.train.qlora import load_config, train

cfg = load_config("configs/train.yaml")
adapter = train(cfg, smoke=False)   # -> outputs/sft-v1
print("adapter:", adapter)

## 5. Base vs tuned on the quarantined hard cases (the midweek gate)

In [ ]:
from src.eval.run import evaluate, compare, write_report, _load_examples
from src.infer import load_hf_tagger

examples = _load_examples("eval/hardcases")
print(f"{len(examples)} quarantined hard-cases examples\n")

base = load_hf_tagger()
rep_base = evaluate(base, examples, label="base")
write_report(rep_base)

tuned = load_hf_tagger(adapter=adapter)
rep_tuned = evaluate(tuned, examples, label="tuned-v1")
write_report(rep_tuned)

print(compare([rep_base, rep_tuned]))
print("\n--- per-category F5 (base -> tuned) ---")
for cat in sorted({e.category for e in examples}):
    b = rep_base.by_category.get(cat); t = rep_tuned.by_category.get(cat)
    if b and t:
        print(f"  {cat:18s} f5 {b.f5:.2f} -> {t.f5:.2f} | over_tag {b.over_tag_rate:.2f} -> {t.over_tag_rate:.2f}")

## 6. Read the gate

**Win condition (SPOV 7):** tuned beats base on the ambiguous cases — especially **over_tag_rate ↓**
and **f5 ↑** on `person_vs_eponym`, `person_vs_place`, `person_vs_common`, while recall stays high.

- If tuned clearly beats base → the core thesis is validated early. Save the table into `docs/results.md`.
- If it doesn't → treat it as signal, investigate **data quality first** (per the falsifiable-bet
  rule): is the CRAPII slice too small / label-mismatched? add the teacher synthetic bulk (Cell 2).

Reports are saved under `data/eval_reports/`. Paste the table back and we'll record it + plan Day 4."